# Trabajo Práctico de Laboratorio Nº 8: Método Diferencial


## Datos del grupo

* **Curso:** R4052
* **Nº de Grupo:** 2
* **Integrantes:**
  1.   Paul Leandro Encinas Claros
  2.   Franco Ezequiel Ferrario
  3.   Matías Sebastián Marchesi
  4.   Bruno Quiroga

  


# Inicialización

Se deja un bloque de código para la inicialización de las funciones e imports de bibliotecas necesarias.

In [14]:
# Librerías de Python a utilizar
import numpy as np
import math
import IPython
from IPython.display import display, Math
import pandas as pd

In [15]:
# Funciones Auxiliares para cálculo de Incertidumbre
def incert_A(mediciones):
    s = np.std(mediciones, ddof=1)
    return s / math.sqrt(len(mediciones))

def incert_B(err_rel, nd, res, valor):
    # Incertidumbre tipo B (distribución rectangular)
    err_limite = (err_rel / 100) * valor + nd * res
    return err_limite / math.sqrt(3)

def incert_combinada(uA, uB):
    return math.sqrt(uA**2 + uB**2)

def incert_expandida(uC, k):
    return k * uC

def formatear_resultado(valor, incertidumbre):
    if incertidumbre == 0:
        return f"({valor} ± 0)"

    # Redondear la incertidumbre a 2 cifras significativas
    magnitud_incert = math.floor(math.log10(abs(incertidumbre)))
    incert_redondeada = round(incertidumbre, -magnitud_incert + 1)

    # Determinar la cantidad de decimales a mostrar
    decimales = max(0, -magnitud_incert + 1)

    # Truncar el valor medido a la misma cantidad de decimales
    factor = 10**decimales
    valor_truncado = math.trunc(valor * factor) / factor

    if decimales == 0:
        return f"({int(valor_truncado)} ± {int(incert_redondeada)})"
    else:
        return f"({valor_truncado:.{decimales}f} ± {incert_redondeada:.{decimales}f})"

# Objetivos e Instrumental Requerido

## Objetivos
* Determinar la resistencia de salida ($R_o$) de un regulador de tensión LM7805 utilizando el Método Diferencial.
* Analizar la mejora en la incertidumbre al medir de forma diferencial la caída de tensión en lugar de utilizar el método de deflexión directa (el cual arrastraría errores severos al restar dos lecturas grandes muy cercanas).

## Instrumental utilizado
* Voltímetro y Amperímetro: UNI-T UT60E utilizados para ambos roles.
* Regulador de tensión: LM7805 con capacitores de desacople ($0.22 \mu F$, $0.1 \mu F$, $10 \mu F$ Tantalio).
* Resistencia de carga: $R_L \approx 6.4 \, \Omega$.
* Reóstato como $R_2$ del circuito de referencia.
* Fuente regulable de tensión ajustada a $V_F = 7.5 \, V$.

# Procedimiento y Ecuaciones del Método Diferencial

## Descripción

Para determinar la resistencia de salida del regulador LM7805 mediante el método diferencial, se utiliza un circuito de medición que permite comparar la tensión de salida del regulador con una tensión de referencia ajustable.

En primer lugar, se ajusta el divisor resistivo para minimizar la diferencia de tensión entre ambas señales con la salida sin carga. Si no es posible alcanzar una diferencia nula, se registra la diferencia residual, denominada offset $V_{os}$.

Al conectar una carga $R_L$, la tensión de salida del regulador experimenta una pequeña variación debido a su resistencia de salida. Esta variación se determina midiendo la diferencia de tensión $\Delta V$ entre la salida del regulador y la referencia. A partir de esta diferencia, corregida por el offset inicial, y de la corriente de carga $I_L$, se calcula la resistencia de salida.

## Ecuaciones teóricas

Considerando el offset medido en vacío $V_{os}$, la variación de tensión atribuible a la conexión de la carga se expresa como:

$$
\Delta V_{\mathrm{real}} = \Delta V - V_{os}
$$

La resistencia de salida del regulador se calcula mediante:

$$
R_o = \frac{\Delta V - V_{os}}{I_L}
$$

Para determinar la incertidumbre de $R_o$, se calculan los coeficientes de sensibilidad respecto de cada una de las magnitudes de entrada:

$$
C_{\Delta V} = \frac{\partial R_o}{\partial \Delta V}
= \frac{1}{I_L}
$$

$$
C_{V_{os}} = \frac{\partial R_o}{\partial V_{os}}
= -\frac{1}{I_L}
$$

$$
C_{I_L} = \frac{\partial R_o}{\partial I_L}
= -\frac{\Delta V - V_{os}}{I_L^2}
$$

Suponiendo que las magnitudes de entrada son independientes, la incertidumbre estándar combinada de la resistencia de salida se obtiene mediante:

$$
u_c(R_o) =
\sqrt{
C_{\Delta V}^2 u^2(\Delta V)
+ C_{V_{os}}^2 u^2(V_{os})
+ C_{I_L}^2 u^2(I_L)
}
$$

Finalmente, la incertidumbre expandida se calcula como:

$$
U(R_o) = k\,u_c(R_o)
$$

donde $k$ es el factor de cobertura adoptado para expresar la incertidumbre del resultado.

## Circuito
Se utilizó el circuito de medición propuesto en la consigna, conectando el regulador LM7805 y la batería de 9 V mediante el método diferencial.

<div style="text-align: center;">
  <img src="recursos/circuito.png" width="600">
</div>

## Procedimiento paso a paso

### Paso 1: Medición sin carga

Con la carga desconectada, se midió con el UT60E, en la escala de milivolts, la diferencia de tensión entre la salida del regulador y la referencia $V_{bat}$. Se ajustó el reóstato hasta obtener el valor más cercano a $0\,\mathrm{V}$. No se logró una diferencia nula, y el valor residual obtenido fue el offset:

$$
V_{os} = 0{,}2\,\mathrm{mV}
$$

### Paso 2: Medición con carga

Sin modificar el reóstato, se conectó la carga $R_L \approx 6{,}4\,\Omega$ y se registraron diez pares de lecturas de $\Delta V$ (voltímetro diferencial) e $I_L$ (amperímetro) para cada ensayo.

### Paso 3: Revalidación

Con la carga desconectada, se volvió a medir la diferencia de tensión sin carga. La indicación volvió a $0{,}2\,\mathrm{mV}$, igual a la del Paso 1, por lo que la medición se considera válida.

## Ensayos realizados

### Ensayo 1: Bien medido

La medición se realizó con los cables azules, soldados directamente a los terminales del regulador, que hacen el camino más corto entre el regulador y la carga.

### Ensayo 2: Mal medido

La medición se realizó directamente desde los otros cables, que no están conectados lo más cerca posible del regulador.

<div style="text-align: center;">
  <img src="recursos/regulador.jpeg" width="350">
</div>

*Regulador LM7805: cables azules soldados a los terminales (bien medido) y cables de protoboard (mal medido).*

## Cálculo

En ambos ensayos se utiliza la misma ecuación de medición, con los valores medios de las diez lecturas:

$$
R_o = \frac{\Delta V - V_{os}}{I_L}
$$

In [16]:
# Parámetros del instrumento y cobertura
k = 2  # Factor de cobertura 95%

# Especificaciones del UT60E como Voltímetro, Rango de milivolts
err_rel_v = 0.8
nd_v = 3
res_v = 0.0001 # Resolución en Volts (0.1 mV)

# Especificaciones del UT60E como Amperímetro, Rango 10A
err_rel_i = 1.2
nd_i = 5
res_i = 0.001 # Resolución en Amperes (1 mA)

def procesar_resistencia_salida(dV_mediciones, I_mediciones, V_os):
    dV_media = np.mean(dV_mediciones)
    I_media = np.mean(I_mediciones)

    # Valor Nominal de Ro
    Ro_medido = (dV_media - V_os) / I_media

    # Incertidumbres Tipo A
    uA_dV = incert_A(dV_mediciones)
    uA_I = incert_A(I_mediciones)

    # Incertidumbres Tipo B
    uB_dV = incert_B(err_rel_v, nd_v, res_v, dV_media)
    uB_Vos = incert_B(err_rel_v, nd_v, res_v, V_os) # El offset tiene su propia uB
    uB_I = incert_B(err_rel_i, nd_i, res_i, I_media)

    # Combinadas por variable
    uC_dV = incert_combinada(uA_dV, uB_dV)
    uC_I = incert_combinada(uA_I, uB_I)

    # Coeficientes de Sensibilidad
    C_dV = 1 / I_media
    C_Vos = -1 / I_media
    C_I = -(dV_media - V_os) / (I_media**2)

    # Incertidumbre Combinada y Expandida de Ro
    uC_Ro = math.sqrt((C_dV * uC_dV)**2 + (C_Vos * uB_Vos)**2 + (C_I * uC_I)**2)
    U_Ro = incert_expandida(uC_Ro, k)

    return Ro_medido, U_Ro, dV_media, I_media

# Resolución matemática y mediciones


In [17]:
# Offset medido sin carga en Volts
V_os = 0.2 / 1000  # 0.2 mV

# Revalidación: valor medido sin carga luego de las mediciones con carga
V_os_revalidacion = 0.2 / 1000  # 0.2 mV

# Ensayo 1: bien medido
dV_ensayo1 = [155.7, 148.7, 137.7, 139.9, 129.3, 129.9, 146.8, 141.5, 152.9, 178.5] #mV
dV_ensayo1 = [x / 1000 for x in dV_ensayo1]
I_ensayo1 = [1.022, 1.022, 1.024, 1.022, 1.022, 1.022, 1.02, 1.017, 1.013, 1.011] #A

# Ensayo 2: mal medido
dV_ensayo2 = [59.7, 55.7, 55.8, 73.3, 54.2, 63.0, 63.0, 69.2, 71.4, 65.3] #mV
dV_ensayo2 = [x / 1000 for x in dV_ensayo2]
I_ensayo2 = [0.962, 0.964, 0.963, 0.963, 0.96, 0.959, 0.99, 0.992, 0.997, 0.997] #A

# Procesamiento
Ro1, U_Ro1, dV1_prom, I1_prom = procesar_resistencia_salida(dV_ensayo1, I_ensayo1, V_os)
Ro2, U_Ro2, dV2_prom, I2_prom = procesar_resistencia_salida(dV_ensayo2, I_ensayo2, V_os)

## Verificación del offset

Se compara el valor medido sin carga antes de las mediciones (Paso 1) con el medido después (Paso 3).

In [18]:
df_offset = pd.DataFrame({
    "Instante": ["Antes de medir (Paso 1)", "Después de medir (Paso 3)"],
    "Diferencia sin carga (mV)": [V_os * 1000, V_os_revalidacion * 1000]
})
display(df_offset)

,Instante,Diferencia sin carga (mV)
0,Antes de medir (Paso 1),0.2
1,Después de medir (Paso 3),0.2


## Mediciones registradas

Se muestran las diez lecturas de cada ensayo: el bien medido (Ensayo 1) y el mal medido (Ensayo 2).

In [19]:
df_mediciones = pd.DataFrame({
    "Nro": range(1, 11),
    "I_L bien medido (A)": I_ensayo1,
    "ΔV bien medido (mV)": [x * 1000 for x in dV_ensayo1],
    "I_L mal medido (A)": I_ensayo2,
    "ΔV mal medido (mV)": [x * 1000 for x in dV_ensayo2],
})
display(df_mediciones.round(3).set_index("Nro"))

,I_L bien medido (A),ΔV bien medido (mV),I_L mal medido (A),ΔV mal medido (mV)
Nro,,,,
1,1.022,155.7,0.962,59.7
2,1.022,148.7,0.964,55.7
3,1.024,137.7,0.963,55.8
4,1.022,139.9,0.963,73.3
5,1.022,129.3,0.960,54.2
6,1.022,129.9,0.959,63.0
7,1.020,146.8,0.990,63.0
8,1.017,141.5,0.992,69.2
9,1.013,152.9,0.997,71.4


# Resultados de las mediciones

En esta sección se resumen los resultados obtenidos

In [21]:
# Resumen de resultados
datos_diferencial = {
    "Ensayo": ["Bien medido", "Mal medido"],
    "I_L Promedio (A)": [I1_prom, I2_prom],
    "ΔV Promedio (mV)": [dV1_prom * 1000, dV2_prom * 1000],
    "Vos (mV)": [V_os * 1000, V_os * 1000],
    "Ro Calculada (Ω)": [Ro1, Ro2],
    "U Expandida (Ω)": [U_Ro1, U_Ro2]
}

df_dif = pd.DataFrame(datos_diferencial)

# Formateo
df_dif["Resultado Ro (Ω)"] = df_dif.apply(
    lambda row: formatear_resultado(row['Ro Calculada (Ω)'], row['U Expandida (Ω)']), axis=1
)

display(df_dif[["Ensayo", "I_L Promedio (A)", "ΔV Promedio (mV)", "Resultado Ro (Ω)"]])

# Comparación entre ambos ensayos
dif_abs = Ro2 - Ro1
dif_rel = dif_abs / Ro1 * 100
U_dif = math.sqrt(U_Ro1**2 + U_Ro2**2)

print(f"Ro bien medido: {Ro1*1000:.1f} mΩ (U = {U_Ro1*1000:.1f} mΩ)")
print(f"Ro mal medido:  {Ro2*1000:.1f} mΩ (U = {U_Ro2*1000:.1f} mΩ)")
print(f"Diferencia: {dif_abs*1000:.1f} mΩ ({dif_rel:.1f} % respecto del bien medido)")
if abs(dif_abs) <= U_dif:
    print("Los resultados son compatibles: Sí")
else:
    print("Los resultados son compatibles: No")

,Ensayo,I_L Promedio (A),ΔV Promedio (mV),Resultado Ro (Ω)
0,Bien medido,1.0195,146.09,(0.1430 ± 0.0095)
1,Mal medido,0.9747,63.06,(0.0644 ± 0.0047)


Ro bien medido: 143.1 mΩ (U = 9.5 mΩ)
Ro mal medido:  64.5 mΩ (U = 4.7 mΩ)
Diferencia: -78.6 mΩ (-54.9 % respecto del bien medido)
Los resultados son compatibles: No


# Conclusiones

1. El offset sin carga fue de $0{,}2\,\text{mV}$ antes y después de medir, por lo que las mediciones son válidas.

2. En el ensayo bien medido se obtuvo $\Delta V \approx 146\,\text{mV}$ con $I_L \approx 1{,}02\,\text{A}$, y $R_o = (0{,}143 \pm 0{,}010)\,\Omega$. En el mal medido se obtuvo $\Delta V \approx 63\,\text{mV}$ con $I_L \approx 0{,}975\,\text{A}$, y $R_o = (0{,}064 \pm 0{,}005)\,\Omega$ ($k=2$).

3. Los resultados difieren en unos $79\,\text{m}\Omega$ (cerca de $55\,\%$), mientras que la incertidumbre expandida de la diferencia es de unos $11\,\text{m}\Omega$. Los dos resultados no son compatibles, por lo que el punto de conexión de la medición respecto del regulador afecta directamente al resultado.

4. El método diferencial permite medir $\Delta V$ directamente en la escala de milivolts, con una incertidumbre mucho menor que la de restar dos lecturas de $5\,\text{V}$.